In [1]:
# Import statements
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from pandas.tseries.offsets import DateOffset

from lightgbm import LGBMRegressor

In [2]:
# Uploading the dataframe
df1 = pd.read_csv('content/idx_cleaned_data1.csv')
df2 = pd.read_csv('content/idx_cleaned_data2.csv', low_memory=False)
df3 = pd.read_csv('content/idx_cleaned_data3.csv', low_memory=False)

In [3]:
df = pd.merge(df1, df2, how='outer')
df = pd.merge(df, df3, how='outer')

In [4]:
# Removing any potential outliers within the ClosePrice variable
Q1 = df['ClosePrice'].quantile(0.005)
Q3 = df['ClosePrice'].quantile(0.995)

df = df[(df['ClosePrice'] >= Q1) & (df['ClosePrice'] <= Q3)]

In [5]:
# Creating the train/test split (time-based)
df['CloseDate'] = pd.to_datetime(df['CloseDate'], yearfirst=True)

last_date  = df['CloseDate'].max()
test_date  = last_date - DateOffset(months=1)
train_date = test_date - DateOffset(months=6)

train_df = df[(df['CloseDate'] > train_date) & (df['CloseDate'] < test_date)]
test_df  = df[df['CloseDate'] >= test_date]
test_df  = test_df[
    (test_df['ClosePrice'] > test_df['ClosePrice'].quantile(0.05)) &
    (test_df['ClosePrice'] < test_df['ClosePrice'].quantile(0.95))
]

train_y = train_df['ClosePrice']
test_y  = test_df['ClosePrice']

In [6]:
# Dropping columns to improve the df
drop_cols = [
    'Flooring', 'Levels', 'UnparsedAddress', 'PostalCode', 'City',
    'StreetNumberNumeric', 'PropertyType', 'PropertySubType',
    'ClosePrice', 'CloseDate', 'DaysOnMarket', 'BuyerOfficeName',
    'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
    'BuyerAgentAOR', 'BuyerOfficeAOR', 'ContractStatusChangeDate',
    'PurchaseContractDate', 'MLSAreaMajor', 'CountyOrParish',
    'ElementarySchool', 'SubdivisionName', 'ListingContractDate',
    'HighSchool', 'HighSchoolDistrict', 'StateOrProvince',
    'MiddleOrJuniorSchool'
]
train_X = train_df.drop(columns=drop_cols)
test_X  = test_df.drop(columns=drop_cols)

In [8]:
# Keep only numeric columns (same logic as original)
num_cols = train_df.select_dtypes(include='number').columns
nf_train_df = train_df.copy()

In [9]:
# ── NEW FEATURES (from original notebook) ──────────────────────────────────
# bath-to-bedroom ratio
nf_train_df['bbratio'] = (
    nf_train_df['BathroomsTotalInteger'] /
    nf_train_df['BedroomsTotal'].replace(0, 1)
).fillna(0)

# property age at contract date
nf_train_df['age'] = (
    pd.to_datetime(nf_train_df['ContractStatusChangeDate']).dt.year
    - nf_train_df['YearBuilt']
)

# Mirror the same new features onto test_df
test_df = test_df.copy()
test_df['bbratio'] = (
    test_df['BathroomsTotalInteger'] /
    test_df['BedroomsTotal'].replace(0, 1)
).fillna(0)
test_df['age'] = (
    pd.to_datetime(test_df['ContractStatusChangeDate']).dt.year
    - test_df['YearBuilt']
)

In [10]:
# Refresh num_cols after adding new features
num_cols    = nf_train_df.select_dtypes(include='number').columns
nf_train_df = nf_train_df[num_cols]

In [12]:
# Prepare training and test arrays — use all numeric columns
X_train = nf_train_df.drop(columns=['ClosePrice'])
y_train = nf_train_df['ClosePrice']

# Align test columns with training columns (fill any missing with 0)
test_num_cols = num_cols  # same set used for training
X_test  = test_df[test_num_cols].drop(columns=['ClosePrice'], errors='ignore')
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
y_test  = test_df['ClosePrice']

print(f"Train shape: {X_train.shape}  |  Test shape: {X_test.shape}")

Train shape: (99165, 19)  |  Test shape: (20375, 19)


In [13]:
param_grid = {
    'n_estimators':    [100, 300, 500],
    'max_depth':       [3, 5, 7],
    'learning_rate':   [0.05, 0.1, 0.2],
    'subsample':       [0.7, 1.0],
    'colsample_bytree':[0.7, 1.0],
}

lgbm_base = LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=1
)

grid_search = GridSearchCV(
    estimator=lgbm_base,
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    verbose=2,
    n_jobs=1
)

grid_search.fit(X_train, y_train)

print("\nBest parameters:", grid_search.best_params_)
print(f"Best CV R²:       {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 108 candidates, totalling 324 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001545 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2981
[LightGBM] [Info] Number of data points in the train set: 66110, number of used features: 19
[LightGBM] [Info] Start training from score 857804.002272
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

In [14]:
best_xgb = grid_search.best_estimator_
y_pred   = best_xgb.predict(X_test)

r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test R²:   {r2:.4f}")
print(f"Test RMSE: {rmse:,.2f}")

Test R²:   0.4004
Test RMSE: 438,564.81
